<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

 <p><font size="5" color='grey'> <b> LCEL Vertiefung &mdash; Brücke zu LangGraph </b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
# Wichtig: LangSmith-Account und API-Key im EU-Workspace anlegen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M07-LCEL-Chains"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---



Diese Modul vertrieft LCEL mit dem Pipe-Operator `|`:




In [ ]:
# Imports für LCEL-Beispiele
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 2 | Was ist LCEL?
---

> 📌 **Grundlagen** (Pipe-Operator, erste Chain) wurden in **M03 Kap 3** eingeführt. Hier folgt die Vertiefung.

LCEL (LangChain Expression Language) beschreibt Pipelines aus **Runnable**-Bausteinen mit klarem Input/Output-Protokoll:

- Jeder Baustein implementiert `.invoke()`, `.stream()`, `.batch()`
- Bausteine sind kombinierbar: `a | b | c` → neue Runnable
- Das Protokoll gilt für: Prompts, LLMs, Parser, Tools, eigene Funktionen (`RunnableLambda`)

LCEL ist gut wartbar, solange die Kette linear bleibt. Sobald Verzweigungen oder bedingte Logik nötig werden, ist LangGraph die bessere Wahl.


In [ ]:
# Mini-Check: Ein einfacher RunnableLambda-Schritt
normalize = RunnableLambda(lambda x: x.strip().lower())
print(normalize.invoke("   HALLO LCEL   "))

# 3 | Pipe-Operator
---

Mit `|` verbinden Sie Schritte zu einer Kette.
Das macht Datenflüsse explizit und gut testbar.


In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Erstelle 3 Stichpunkte zu folgendem Thema: {topic}"
)

basic_chain = prompt | llm | parser

result = basic_chain.invoke({"topic": "Vorteile von Unit Tests"})

mprint('### Ergebnis')
mprint('---')
mprint(result)

# 4 | Minimalbeispiel: prompt | llm
---



Ohne Parser erhalten Sie ein AIMessage-Objekt.
Mit `StrOutputParser()` arbeiten Sie direkt mit Text weiter.


In [ ]:
chain_raw = prompt | llm
raw = chain_raw.invoke({"topic": "Python Dataclasses"})

mprint('### Ergebnis')
mprint('---')
mprint(raw.content)

# 5 | Komplexe Chains
---

`RunnableParallel` erlaubt parallele Teilketten.
So lassen sich z. B. Zusammenfassung und Kritik gleichzeitig erzeugen.


In [ ]:
base_prompt = ChatPromptTemplate.from_template(
    "Analysiere den folgenden Text: {text}"
)

summary_chain = (
    base_prompt
    | llm
    | parser
    | RunnableLambda(lambda t: f"Zusammenfassung:\n{t}")
)

critic_chain = (
    ChatPromptTemplate.from_template(
        "Nenne 3 mögliche Schwächen in diesem Text:\n{text}"
    )
    | llm
    | parser
    | RunnableLambda(lambda t: f"Kritik:\n{t}")
)

combined_chain = RunnableParallel(
    summary=summary_chain,
    critique=critic_chain,
)

text = "LCEL ist praktisch, weil Chains deklarativ und modular aufgebaut werden können."
out = combined_chain.invoke({"text": text})

mprint('### Ergebnis')
mprint('---')
mprint(out["summary"])
mprint('---')
mprint(out["critique"])

# 6 | Chain-Debugging mit LangSmith
---

Mit LangSmith sehen Sie:
- welche Chain-Schritte aufgerufen wurden
- welche Inputs/Outputs pro Schritt entstanden sind
- wo Fehler oder unerwartete Antworten auftreten


<p><font color='black' size="5">
LangSmith konfigurieren
</font></p>

**Hinweis:** Verwenden Sie den EU-API-Endpoint `https://eu.api.smith.langchain.com`.

<p><font color='black' size="5">
Tracing
</font></p>

In [ ]:
# Beispiel-Komponenten (falls noch nicht definiert)
prompt = ChatPromptTemplate.from_template("Erzähle mir kurz etwas über {topic}")

# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M07_Kap6_Chain_Debugging",
    "tags":     ["M07", "lcel", "chain", "debug"]
}

# 2. with_config() anwenden, dann invoke()
debug_chain = (prompt | llm | parser).with_config(**run_cfg)
dbg = debug_chain.invoke({"topic": "Wann lohnt sich Caching?"})

mprint("### Trace erstellt")
mprint('---')
mprint(dbg)

# 7 | Streaming mit LCEL

- Unterschied `invoke()` vs. `stream()`/`astream()`
- Wann Streaming UX verbessert (lange Antworten, Live-Feedback)
- Mini-Demo: gleiche Chain einmal normal, einmal gestreamt

---


In [ ]:
stream_prompt = ChatPromptTemplate.from_template(
    "Erkläre das Konzept '{topic}' in 5 kurzen Absätzen."
)

run_cfg = {"run_name": "M07_Kap7_Streaming", "tags": ["M07", "streaming"]}
stream_chain = (stream_prompt | llm | parser).with_config(**run_cfg)

mprint("### Normaler Aufruf (invoke):")
mprint('---')
full = stream_chain.invoke({"topic": "Event-getriebene Agenten"})
mprint(full)

In [ ]:
mprint("### Gestreamter Aufruf (stream):")
mprint('---')
for chunk in stream_chain.stream({"topic": "Event-getriebene Agenten"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M07-LCEL-Chains", limit=3, show_steps=True)

# 8 | Ausblick: Brücke zu LangGraph
---

LCEL-Chains sind linear — jeder Schritt folgt dem nächsten. **LangGraph** löst die eine Einschränkung, die Chains haben:
**keine Schleifen, keine Verzweigungen**.

| LCEL-Konzept | LangGraph-Äquivalent | Unterschied |
|---|---|---|
| `prompt \| llm \| parser` | Sequenz von Nodes | LangGraph kann zurückspringen |
| `RunnableParallel` | Parallele Edges | LangGraph steuert den Fluss dynamisch |
| `RunnableLambda` | Node-Funktion | Nodes haben Zugriff auf den gesamten State |
| `chain.invoke()` | `graph.invoke()` | Identische API |

```
LCEL:       A → B → C → Ende          (immer linear)
LangGraph:  A → B → C → A (Schleife)  (bedingte Kanten möglich)
            A → B oder D  (Verzweigung)
```

> 💡 **Alles, was Sie über Chains gelernt haben, gilt in LangGraph weiter.**  
> Der Unterschied: Nodes können den nächsten Schritt selbst entscheiden — das ist **M08**.

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgabe eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**
- Bauen Sie eine einfache LCEL-Chain mit Prompt -> LLM -> Parser.
- Testen Sie die Chain mit einem Beispielinput.

In [ ]:
# ✅ Selbstcheck Grundlagen
# Voraussetzung: Einfache LCEL-Chain (Prompt | LLM | Parser) in 'meine_chain' gespeichert.

_chain = meine_chain  # ← Variablennamen anpassen

assert hasattr(_chain, "invoke"), \
    "❌ Chain hat kein invoke() – wurde | (Pipe-Operator) korrekt verwendet?"

_input_key = "thema"  # ← Input-Key deines PromptTemplates anpassen (z.B. 'text', 'eingabe')
_r = _chain.invoke({_input_key: "KI-Agenten"})
assert _r is not None and len(str(_r)) > 10, \
    "❌ Ausgabe leer oder zu kurz – prüfe StrOutputParser und LLM-Verbindung"

# Grundlegende Strukturprüfung: Chain muss aus mindestens 3 Schritten bestehen (Prompt | LLM | Parser)
_steps = getattr(_chain, "steps", None)
assert _steps is not None and len(_steps) >= 3, \
    f"❌ Chain hat nur {len(_steps) if _steps else '?'} Schritte – Grundlagen erfordern mindestens: Prompt | LLM | Parser (3 Schritte)"

print(f"✅ Grundlagen-Selbstcheck bestanden! (Chain: {len(_steps)} Schritte · Ausgabe-Typ: {type(_r).__name__})")
print(f"   {str(_r)[:120]}{'...' if len(str(_r)) > 120 else ''}")

**Aufbau**
1. Bauen Sie eine LCEL-Chain mit mindestens 4 Schritten (Prompt -> LLM -> Parser -> Postprocessing).
2. Implementieren Sie eine kleine Bewertungsfunktion (`RunnableLambda`), die die Antwortlaenge oder Struktur prueft.
3. Testen Sie die Chain mit 3 unterschiedlichen Inputs und notieren Sie Auffaelligkeiten.

In [ ]:
# ✅ Selbstcheck Aufbau — 4-Schritt-Chain mit Postprocessing
# Voraussetzung: LCEL-Chain mit >= 4 Schritten (Prompt | LLM | Parser | RunnableLambda)
# in 'meine_aufbau_chain' gespeichert.
# Beispiel: meine_aufbau_chain = prompt | llm | parser | RunnableLambda(lambda t: t.strip())

from langchain_core.runnables import RunnableLambda, RunnableSequence

assert 'meine_aufbau_chain' in dir() or 'meine_aufbau_chain' in locals(), \
    "❌ Variable 'meine_aufbau_chain' fehlt – speichern Sie Ihre 4-Schritt-Chain in 'meine_aufbau_chain'"
assert hasattr(meine_aufbau_chain, "invoke"), \
    "❌ 'meine_aufbau_chain' hat kein invoke() – Pipe-Operator korrekt verwendet?"

# Schritte zählen
_aufbau_steps = getattr(meine_aufbau_chain, "steps", None)
assert _aufbau_steps is not None and len(_aufbau_steps) >= 4, \
    f"❌ Chain hat nur {len(_aufbau_steps) if _aufbau_steps else '?'} Schritte – Aufbau erfordert >= 4 Schritte (Prompt | LLM | Parser | Postprocessing)"

# RunnableLambda als Postprocessing-Schritt prüfen
_hat_lambda = any(isinstance(s, RunnableLambda) for s in _aufbau_steps)
assert _hat_lambda, \
    "❌ Kein RunnableLambda-Schritt gefunden – Aufbau erfordert eine Bewertungs-/Postprocessing-Funktion als RunnableLambda"

# Funktionstest mit 3 Inputs
_test_inputs = ["Unit Tests", "Caching", "Event-getriebene Systeme"]
_test_input_key = "thema"  # ← Input-Key anpassen
_ergebnisse = [meine_aufbau_chain.invoke({_test_input_key: inp}) for inp in _test_inputs]
assert all(_ergebnisse) and all(len(str(e)) > 10 for e in _ergebnisse), \
    "❌ Mindestens ein Test-Ergebnis ist leer – prüfe LLM-Verbindung und Parser"

print(f"✅ Aufbau-Selbstcheck bestanden! ({len(_aufbau_steps)} Schritte · RunnableLambda vorhanden · 3/3 Tests ok)")

**Vertiefung**
4. Ergaenzen Sie eine zweite Teilkette und kombinieren Sie beide mit `RunnableParallel`.

In [ ]:
# ✅ Selbstcheck Vertiefung — RunnableParallel mit zwei Teilketten
# Voraussetzung: Kombinierte Chain mit RunnableParallel in 'meine_parallel_chain' gespeichert.
# Beispiel: meine_parallel_chain = RunnableParallel(summary=chain_a, critique=chain_b)

from langchain_core.runnables import RunnableParallel

assert 'meine_parallel_chain' in dir() or 'meine_parallel_chain' in locals(), \
    "❌ Variable 'meine_parallel_chain' fehlt – kombinieren Sie zwei Teilketten mit RunnableParallel"
assert isinstance(meine_parallel_chain, RunnableParallel), \
    f"❌ 'meine_parallel_chain' ist kein RunnableParallel (Typ: {type(meine_parallel_chain).__name__})"

# Mindestens 2 parallele Teilketten
_parallel_steps = meine_parallel_chain.steps
assert len(_parallel_steps) >= 2, \
    f"❌ Nur {len(_parallel_steps)} Teilkette(n) – RunnableParallel benötigt >= 2 parallele Ketten"

# Funktionstest
_test_input_key_p = "text"  # ← Input-Key anpassen
_pr = meine_parallel_chain.invoke({_test_input_key_p: "LCEL macht Pipelines modular und testbar."})
assert isinstance(_pr, dict), \
    "❌ Ausgabe von RunnableParallel muss ein Dict sein"
assert len(_pr) >= 2, \
    f"❌ Dict hat nur {len(_pr)} Eintrag/-äge – erwartet mind. 2 (je eine Teilkette)"
assert all(isinstance(v, str) and len(v) > 10 for v in _pr.values()), \
    "❌ Mindestens ein Teilketten-Ergebnis ist leer – prüfe beide Teilketten"

import os
assert os.environ.get("LANGSMITH_TRACING") == "true", \
    "❌ LangSmith-Tracing nicht aktiv – für die Vertiefung muss das Tracing eingeschaltet sein"

print(f"✅ Vertiefung-Selbstcheck bestanden! (RunnableParallel · {len(_pr)} Teilketten · LangSmith aktiv)")
print(f"   Schlüssel: {', '.join(_pr.keys())}")